# 📚 BookBot — Assistente Virtual da PageStore

Agente conversacional com IA para uma livraria online fictícia.

**Execute as células em ordem, de cima para baixo.**

---
| Campo | Valor |
|---|---|
| Agente | BookBot |
| Loja | PageStore |
| Modelo | gpt-4o-mini (OpenAI) |
| Interface | Chat interativo no Colab |

## Célula 1 — Instalação de dependências

In [ ]:
!pip install openai==1.30.1 --quiet
print('✅ Dependências instaladas!')

## Célula 2 — Configuração da chave da OpenAI

> **Opção recomendada:** use os Secrets do Colab 🔑
> 1. Clique no ícone de cadeado na barra lateral esquerda
> 2. Adicione um secret com o nome `OPENAI_API_KEY` e sua chave como valor
> 3. Ative o acesso ao notebook e execute esta célula

In [ ]:
import os

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print('✅ Chave carregada via Secrets do Colab!')
except Exception:
    # Alternativa: cole sua chave abaixo
    OPENAI_API_KEY = 'sk-cole-sua-chave-aqui'
    print('⚠️  Chave definida manualmente.')

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

if OPENAI_API_KEY.startswith('sk-'):
    print(f'🔑 Chave configurada: {OPENAI_API_KEY[:8]}...')
else:
    print('❌ Chave inválida — verifique e tente novamente.')

## Célula 3 — Base de Conhecimento: Catálogo de Livros

> Edite `CATALOGO` para adicionar, remover ou atualizar livros.

In [ ]:
# ─── CATÁLOGO DE LIVROS ─────────────────────────────────────────────────────

CATALOGO = [
    {
        'id': 'senhor-dos-aneis',
        'titulo': 'O Senhor dos Anéis',
        'autor': 'J.R.R. Tolkien',
        'genero': ['fantasia', 'aventura', 'épico'],
        'formato': 'Box com 3 volumes (capa dura)',
        'editora': 'HarperCollins Brasil',
        'paginas': 1216,
        'preco': 189.90,
        'parcelas': 6,
        'valor_parcela': 31.65,
        'estoque': 'disponivel',
        'descricao': 'A trilogia épica completa de Tolkien — clássico absoluto da fantasia.',
        'palavras_chave': ['tolkien', 'anel', 'hobbit', 'fantasia epica', 'classico'],
    },
    {
        'id': 'habitos-atomicos',
        'titulo': 'Hábitos Atômicos',
        'autor': 'James Clear',
        'genero': ['autoajuda', 'desenvolvimento pessoal', 'produtividade'],
        'formato': 'Brochura',
        'editora': 'Alta Books',
        'paginas': 320,
        'preco': 49.90,
        'parcelas': 3,
        'valor_parcela': 16.63,
        'estoque': 'disponivel',
        'descricao': 'Guia prático para construir bons hábitos com base em ciência comportamental.',
        'palavras_chave': ['habitos', 'produtividade', 'autoajuda', 'james clear'],
    },
    {
        'id': 'dom-casmurro',
        'titulo': 'Dom Casmurro',
        'autor': 'Machado de Assis',
        'genero': ['literatura brasileira', 'classico', 'romance'],
        'formato': 'Brochura',
        'editora': 'Martin Claret',
        'paginas': 256,
        'preco': 29.90,
        'parcelas': 1,
        'valor_parcela': 29.90,
        'estoque': 'disponivel',
        'descricao': 'O clássico de Machado de Assis — uma das maiores obras da literatura brasileira.',
        'palavras_chave': ['machado de assis', 'classico', 'literatura brasileira', 'capitu'],
    },
    {
        'id': 'clean-code',
        'titulo': 'Clean Code — O Código Limpo',
        'autor': 'Robert C. Martin',
        'genero': ['tecnologia', 'programacao', 'boas praticas'],
        'formato': 'Brochura',
        'editora': 'Alta Books',
        'paginas': 464,
        'preco': 89.90,
        'parcelas': 4,
        'valor_parcela': 22.48,
        'estoque': 'ultimas_unidades',
        'descricao': 'Manual essencial para programadores sobre código legível e sustentável.',
        'palavras_chave': ['clean code', 'programacao', 'codigo', 'robert martin', 'ti'],
    },
    {
        'id': 'harry-potter',
        'titulo': 'Harry Potter e a Pedra Filosofal',
        'autor': 'J.K. Rowling',
        'genero': ['fantasia', 'infanto-juvenil', 'aventura'],
        'formato': 'Capa dura — Edição Ilustrada',
        'editora': 'Rocco',
        'paginas': 272,
        'preco': 79.90,
        'parcelas': 4,
        'valor_parcela': 19.98,
        'estoque': 'disponivel',
        'descricao': 'A edição ilustrada do primeiro livro da saga mais amada do mundo.',
        'palavras_chave': ['harry potter', 'rowling', 'hogwarts', 'fantasia', 'crianca', 'presente'],
    },
]


def formatar_livro(livro):
    estoque = '⚠️ Últimas unidades!' if livro['estoque'] == 'ultimas_unidades' else '✅ Disponível'
    parcela = ''
    if livro['parcelas'] > 1:
        parcela = f" ou {livro['parcelas']}x de R$ {livro['valor_parcela']:.2f} sem juros"
    return (
        f"📖 {livro['titulo']} — {livro['autor']}\n"
        f"Gênero: {', '.join(livro['genero'])}\n"
        f"Formato: {livro['formato']} | {livro['paginas']} páginas\n"
        f"Preço: R$ {livro['preco']:.2f}{parcela}\n"
        f"{estoque}\n"
        f"{livro['descricao']}"
    )


def buscar_livros(query):
    q = query.lower()
    return [
        l for l in CATALOGO
        if any(q in g for g in l['genero'])
        or any(q in p for p in l['palavras_chave'])
        or q in l['titulo'].lower()
        or q in l['autor'].lower()
    ]


def buscar_por_orcamento(maximo):
    return [l for l in CATALOGO if l['preco'] <= maximo]


def listar_catalogo_texto():
    disponiveis = [l for l in CATALOGO if l['estoque'] != 'esgotado']
    return '\n\n─────────────────\n\n'.join(formatar_livro(l) for l in disponiveis)


print(f'✅ Catálogo carregado com {len(CATALOGO)} livros!')

## Célula 4 — Políticas da Loja e FAQ

> Perguntas com os gatilhos definidos aqui são respondidas **sem custo de API**.

In [ ]:
# ─── POLÍTICAS DA PAGESTORE ─────────────────────────────────────────────────

POLITICAS = {
    'frete': {
        'gratis_acima': 99.00,
        'prazo_capital': '3 a 7 dias úteis',
        'prazo_interior': '5 a 12 dias úteis',
        'expresso_preco': 19.90,
        'expresso_prazo': '1 a 2 dias úteis',
    },
    'pagamento': {
        'cartao_parcelas_max': 6,
        'cartao_bandeiras': 'Visa, Mastercard, Elo, Amex',
        'pix_desconto': '10%',
        'boleto_desconto': '10%',
        'debito_desconto': '5%',
    },
    'devolucao': {
        'prazo_desistencia': '7 dias corridos após recebimento',
        'prazo_defeito': '30 dias',
        'prazo_reembolso': '10 dias úteis',
        'condicoes': 'Livro sem uso, na embalagem original e com nota fiscal',
    },
}

FAQ = [
    {
        'gatilhos': ['frete', 'entrega', 'prazo', 'envio', 'demora'],
        'resposta': (
            'Para capitais: 3 a 7 dias úteis. Para o interior: 5 a 12 dias úteis. '
            'Frete expresso por R$ 19,90 (1 a 2 dias úteis). '
            'Frete grátis para compras acima de R$ 99,00! 📦'
        ),
    },
    {
        'gatilhos': ['parcel', 'pagamento', 'pix', 'boleto', 'cartao', 'cartão', 'credito'],
        'resposta': (
            'Aceitamos cartão de crédito em até 6x sem juros (Visa, Mastercard, Elo, Amex), '
            'PIX com 10% de desconto, boleto com 10% de desconto e débito com 5% de desconto. 💳'
        ),
    },
    {
        'gatilhos': ['troc', 'devolu', 'arrependimento', 'cancelar'],
        'resposta': (
            'Você tem 7 dias após o recebimento para devolver sem precisar dar explicações. '
            'O livro deve estar sem uso, na embalagem original e com nota fiscal. '
            'Defeito de fábrica: troca em até 30 dias. Reembolso em até 10 dias úteis. 🔄'
        ),
    },
    {
        'gatilhos': ['original', 'garantia', 'falso', 'pirata'],
        'resposta': (
            'Todos os nossos livros são 100% originais, com nota fiscal e garantia da editora. ✅'
        ),
    },
    {
        'gatilhos': ['rastrear', 'rastreamento', 'onde está', 'onde esta', 'pedido'],
        'resposta': (
            'Você recebe o código de rastreamento por e-mail assim que o pedido for despachado. '
            'Também é possível acompanhar pelo site da PageStore. 📬'
        ),
    },
]


def buscar_faq(mensagem):
    msg = mensagem.lower()
    for item in FAQ:
        if any(g in msg for g in item['gatilhos']):
            return item['resposta']
    return None


print('✅ Políticas e FAQ carregados!')

## Célula 5 — Prompts do BookBot

> Edite `build_system_prompt()` para ajustar a personalidade e regras do agente.

In [ ]:
MENSAGEM_BOAS_VINDAS = """Olá! 👋 Seja bem-vindo(a) à PageStore!
Eu sou o BookBot, seu assistente virtual de livros e leitura. 📚

Como posso te ajudar hoje?

1️⃣  Quero conhecer os livros disponíveis
2️⃣  Preciso de ajuda para escolher um livro
3️⃣  Tenho dúvidas sobre pagamento ou entrega
4️⃣  Outro assunto"""

MENSAGEM_ENCERRAMENTO = """Foi um prazer te atender! 😊
Se precisar de mais alguma coisa, é só chamar.
A PageStore está sempre aqui para você!

Boa leitura e até a próxima! 📖✨"""

GATILHOS_ESCALADA   = ['atendente', 'humano', 'pessoa', 'gerente', 'responsavel', 'responsável']
GATILHOS_ENCERRAMENTO = ['tchau', 'obrigado', 'obrigada', 'até mais', 'ate mais', 'valeu', 'encerrar']


def build_system_prompt():
    catalogo_texto = listar_catalogo_texto()
    p = POLITICAS
    politicas_texto = (
        f"FRETE: grátis acima de R$ {p['frete']['gratis_acima']:.2f} | "
        f"capitais {p['frete']['prazo_capital']} | interior {p['frete']['prazo_interior']} | "
        f"expresso R$ {p['frete']['expresso_preco']:.2f} em {p['frete']['expresso_prazo']}\n"
        f"PAGAMENTO: cartão até {p['pagamento']['cartao_parcelas_max']}x sem juros | "
        f"PIX {p['pagamento']['pix_desconto']} desconto | boleto {p['pagamento']['boleto_desconto']} desconto\n"
        f"DEVOLUÇÃO: {p['devolucao']['prazo_desistencia']} | reembolso em {p['devolucao']['prazo_reembolso']}\n"
        f"Condições: {p['devolucao']['condicoes']}"
    )
    return f"""Você é o BookBot, assistente virtual de vendas da PageStore — uma livraria online.
Ajude clientes a encontrar o livro ideal, responda dúvidas sobre preços, pagamento, frete e políticas.

PERSONALIDADE:
- Seja simpático, entusiasmado com livros e paciente
- Use linguagem clara e acolhedora — como um livreiro apaixonado
- Seja proativo: sugira títulos relacionados
- Responda sempre em português brasileiro

REGRAS:
- NUNCA invente preços, autores ou políticas fora da base de conhecimento
- Nunca processe pagamentos diretamente
- Para assuntos fora do escopo, redirecione gentilmente

FLUXO DE RECOMENDAÇÃO:
1. Pergunte o gênero/tema preferido
2. Pergunte se é presente ou para si mesmo
3. Pergunte o orçamento
4. Recomende o título mais adequado

══════════ CATÁLOGO ══════════
{catalogo_texto}

══════════ POLÍTICAS ══════════
{politicas_texto}"""


print('✅ Prompts configurados!')

## Célula 6 — Memória de Sessão

In [ ]:
MAX_MENSAGENS = 20  # janela deslizante — evita estouro do context window

class Sessao:
    def __init__(self):
        self.historico = []
        self.primeiro_contato = True
        self.total_mensagens = 0
        self.total_tokens = 0

    def adicionar(self, role, content):
        self.historico.append({'role': role, 'content': content})
        if len(self.historico) > MAX_MENSAGENS:
            self.historico = self.historico[-MAX_MENSAGENS:]
        self.primeiro_contato = False
        if role == 'user':
            self.total_mensagens += 1

    def resetar(self):
        self.__init__()

    def estatisticas(self):
        return f'💬 Mensagens: {self.total_mensagens} | 📦 Tokens usados: {self.total_tokens}'


print('✅ Gerenciador de sessão pronto!')

## Célula 7 — Motor do Agente

> Aqui fica a lógica central: FAQ local → OpenAI.

In [ ]:
from openai import OpenAI

MODELO      = 'gpt-4o-mini'
MAX_TOKENS  = 500
TEMPERATURA = 0.7

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])


def deve_escalar(msg):
    return any(g in msg.lower() for g in GATILHOS_ESCALADA)


def deve_encerrar(msg):
    return any(g in msg.lower() for g in GATILHOS_ENCERRAMENTO)


def processar_mensagem(sessao, mensagem_usuario):
    """
    Fluxo:
      1. Primeiro contato  → boas-vindas (sem custo de API)
      2. Encerramento      → mensagem de despedida
      3. Escalada          → redireciona para humano
      4. FAQ local         → resposta direta (sem custo de API)
      5. OpenAI            → processa com contexto completo
    """
    if sessao.primeiro_contato:
        sessao.adicionar('user', mensagem_usuario)
        sessao.adicionar('assistant', MENSAGEM_BOAS_VINDAS)
        return MENSAGEM_BOAS_VINDAS, 'boas_vindas'

    if deve_encerrar(mensagem_usuario):
        sessao.adicionar('user', mensagem_usuario)
        sessao.adicionar('assistant', MENSAGEM_ENCERRAMENTO)
        return MENSAGEM_ENCERRAMENTO, 'encerramento'

    if deve_escalar(mensagem_usuario):
        resp = 'Claro! Vou te conectar com um de nossos especialistas. Um momento! 😊\n[🔴 ESCALADA PARA ATENDIMENTO HUMANO]'
        sessao.adicionar('user', mensagem_usuario)
        sessao.adicionar('assistant', resp)
        return resp, 'escalada'

    faq = buscar_faq(mensagem_usuario)
    if faq:
        sessao.adicionar('user', mensagem_usuario)
        sessao.adicionar('assistant', faq)
        return faq, 'faq'

    # Enriquecimento de contexto com livros relevantes
    livros = buscar_livros(mensagem_usuario)
    contexto_extra = ''
    if livros:
        contexto_extra = '\n\n[Livros relevantes:]\n' + '\n---\n'.join(formatar_livro(l) for l in livros)

    mensagens = [
        {'role': 'system', 'content': build_system_prompt() + contexto_extra},
        *sessao.historico,
        {'role': 'user', 'content': mensagem_usuario},
    ]

    resultado = client.chat.completions.create(
        model=MODELO,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURA,
        messages=mensagens,
    )

    resposta = resultado.choices[0].message.content.strip()
    sessao.total_tokens += resultado.usage.total_tokens
    sessao.adicionar('user', mensagem_usuario)
    sessao.adicionar('assistant', resposta)

    return resposta, 'openai'


print('✅ Motor do BookBot pronto!')

## Célula 8 — ▶️ Iniciar o Chat

> Execute esta célula e converse com o BookBot direto aqui no Colab!
> 
> - Digite normalmente e pressione **Enter** para enviar
> - `catalogo` — lista todos os livros disponíveis
> - `reset` — reinicia a conversa
> - `sair` — encerra o chat

In [ ]:
import textwrap

LARGURA = 65
ICONES  = {'boas_vindas': '👋', 'faq': '📋', 'openai': '🤖', 'encerramento': '👋', 'escalada': '👤'}

def linha(c='─', n=LARGURA): print(c * n)

def exibir_bot(texto, fonte):
    print(f"\n{ICONES.get(fonte,'🤖')} BookBot [{fonte}]:")
    linha('─', 45)
    for l in texto.split('\n'):
        print(textwrap.fill(l, width=LARGURA) if l.strip() else '')
    linha('─', 45)


# ── Inicia sessão ────────────────────────────────────────────────────────────
sessao = Sessao()
linha('═')
print('📚  BookBot — PageStore'.center(LARGURA))
linha('═')
print('Comandos: "sair" | "reset" | "catalogo"')
linha()

resposta, fonte = processar_mensagem(sessao, 'inicio')
exibir_bot(resposta, fonte)

# ── Loop do chat ─────────────────────────────────────────────────────────────
while True:
    try:
        entrada = input('\n✏️  Você: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\nChat encerrado.')
        break

    if not entrada:
        continue

    if entrada.lower() == 'sair':
        exibir_bot(MENSAGEM_ENCERRAMENTO, 'encerramento')
        print(f'\n{sessao.estatisticas()}')
        break

    if entrada.lower() == 'reset':
        sessao.resetar()
        print('\n🔄 Conversa reiniciada!')
        resposta, fonte = processar_mensagem(sessao, 'inicio')
        exibir_bot(resposta, fonte)
        continue

    if entrada.lower() == 'catalogo':
        print('\n📚 Catálogo completo da PageStore:\n')
        print(listar_catalogo_texto())
        continue

    print(f'\n👤 Você: {entrada}')

    try:
        resposta, fonte = processar_mensagem(sessao, entrada)
        exibir_bot(resposta, fonte)
        if fonte in ('escalada', 'encerramento'):
            print(f'\n{sessao.estatisticas()}')
            break
    except Exception as e:
        print(f'\n❌ Erro: {e}')
        print('Verifique sua chave OpenAI na Célula 2 e tente novamente.')

## Célula 9 — Testes Automatizados

> Execute para verificar se todas as funções estão funcionando corretamente.

In [ ]:
def rodar_teste(nome, condicao):
    status = '✅ PASSOU' if condicao else '❌ FALHOU'
    print(f'  {status}  {nome}')
    return condicao

resultados = []
linha('═')
print('🧪  TESTES AUTOMATIZADOS — BookBot'.center(LARGURA))
linha('═')

print('\n📖 Catálogo:')
resultados += [
    rodar_teste('Catálogo tem 5 livros', len(CATALOGO) == 5),
    rodar_teste('Todos têm título', all('titulo' in l for l in CATALOGO)),
    rodar_teste('Todos têm preço numérico', all(isinstance(l['preco'], (int, float)) for l in CATALOGO)),
    rodar_teste('buscar_livros("fantasia") retorna resultado', len(buscar_livros('fantasia')) > 0),
    rodar_teste('buscar_livros("tolkien") acha O Senhor dos Anéis', any(l['id'] == 'senhor-dos-aneis' for l in buscar_livros('tolkien'))),
    rodar_teste('buscar_por_orcamento(50) retorna apenas livros <= R$50', all(l['preco'] <= 50 for l in buscar_por_orcamento(50))),
    rodar_teste('formatar_livro retorna string com título', CATALOGO[0]['titulo'] in formatar_livro(CATALOGO[0])),
]

print('\n📋 FAQ:')
resultados += [
    rodar_teste('FAQ responde sobre frete', buscar_faq('qual o frete?') is not None),
    rodar_teste('FAQ responde sobre pix', buscar_faq('aceita pix?') is not None),
    rodar_teste('FAQ responde sobre troca', buscar_faq('como faço uma troca?') is not None),
    rodar_teste('FAQ retorna None sem gatilho', buscar_faq('qual seu nome?') is None),
    rodar_teste('FAQ sobre frete menciona "dias"', 'dias' in (buscar_faq('frete') or '').lower()),
]

print('\n💾 Sessão:')
s = Sessao()
resultados += [
    rodar_teste('Inicia com primeiro_contato=True', s.primeiro_contato),
]
s.adicionar('user', 'oi')
resultados += [
    rodar_teste('Após mensagem, primeiro_contato=False', not s.primeiro_contato),
    rodar_teste('Histórico tem 1 mensagem', len(s.historico) == 1),
]
s.resetar()
resultados += [
    rodar_teste('Após resetar, histórico vazio', len(s.historico) == 0),
    rodar_teste('Após resetar, primeiro_contato=True', s.primeiro_contato),
]

print('\n🔍 Detecção de intenções:')
resultados += [
    rodar_teste('deve_escalar detecta "atendente"', deve_escalar('quero falar com um atendente')),
    rodar_teste('deve_escalar não dispara em msg normal', not deve_escalar('quero um livro')),
    rodar_teste('deve_encerrar detecta "tchau"', deve_encerrar('tchau, obrigada!')),
    rodar_teste('deve_encerrar não dispara em msg normal', not deve_encerrar('quero um livro')),
]

print('\n💬 Prompt:')
prompt = build_system_prompt()
resultados += [
    rodar_teste('System prompt é string', isinstance(prompt, str)),
    rodar_teste('Contém BookBot', 'BookBot' in prompt),
    rodar_teste('Contém título de livro', 'Hábitos Atômicos' in prompt),
    rodar_teste('Contém política de frete', 'Frete' in prompt),
]

total  = len(resultados)
passou = sum(resultados)
linha('═')
print(f'  Resultado: {passou}/{total} testes passaram')
print('  🎉 Todos os testes passaram! BookBot está pronto.' if passou == total else f'  ⚠️  {total-passou} teste(s) falharam.')
linha('═')

## Célula 10 — Simulação de Conversa Completa

> Executa um roteiro de mensagens automaticamente para demonstração.
> Útil para testar o bot sem precisar digitar manualmente.

In [ ]:
import time

ROTEIRO = [
    'oi',
    'qual o prazo de entrega para o interior?',
    'aceita pix?',
    'quero um livro de fantasia até R$ 100',
    'pode me falar mais sobre o Harry Potter?',
    'obrigada, tchau!',
]

s_sim = Sessao()
linha('═')
print('🎬  SIMULAÇÃO DE CONVERSA'.center(LARGURA))
linha('═')

for msg in ROTEIRO:
    print(f'\n👤 Cliente: {msg}')
    try:
        resposta, fonte = processar_mensagem(s_sim, msg)
        exibir_bot(resposta, fonte)
        if fonte in ('escalada', 'encerramento'):
            break
        time.sleep(1)  # evita rate limit da OpenAI
    except Exception as e:
        print(f'❌ Erro: {e}')
        break

linha('═')
print(s_sim.estatisticas())
linha('═')